<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Query an Iceberg Table and Its Files</div>
</div>

The ten historical orders of Module 1 are also in an Iceberg table in a small local data lake: Silo, an S3-compatible object store, holds its files, and an Iceberg REST Catalog service holds its metadata. You connect Doris to it with a Catalog, find the table, read its snapshots and data files, query it, join it with a customer table in Doris, and load it into an internal table: once through the Catalog, once from its data file with the S3 table function. Every path must give ten orders and 12220.60. Plan about 30 minutes.

**Module 4: Querying External Data and Joining Lake Tables** · [Lesson](course.md) · [Quiz](quiz04_internal_files_and_lake_tables.ipynb) · [Course README](../../README.md)

## Prepare the lake

Complete Lab 1 first. The next cell connects to the course Doris and starts two course containers, Silo and the Iceberg REST Catalog, on the local ports 51900 and 51818 (variables `DW_LAKE_S3_PORT` and `DW_LAKE_REST_PORT`); it checks both ports first, and if one is in use it stops, names the port, and says how to free it or choose another one. The first start downloads the two images; later runs reuse the data in their volumes. The course README's [Clean up](../../README.md#clean-up) section says how to stop and remove them.

The preparation creates, in Doris, a Catalog named after your lab database with `_lake` added, and in it an Iceberg table `orders`; on the first run it inserts the ten orders. **Doris writes this lake table**: the course has no other engine. In a real data lake, tables like this are usually written by Spark, Flink, or another system, and Doris reads them through the same kind of Catalog. When it finishes, the cell shows the SQL it ran. The Lab rebuilds only the internal tables `customers_sample`, `orders_from_lake`, and `orders_from_file`.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.lakehouse import prepare_lakehouse
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()
source = prepare_lakehouse(lab, start=True)

**Expected result**

A panel with six green steps and the check of the ten orders in the lake, then the SQL that the preparation ran: `CREATE CATALOG IF NOT EXISTS dw_course_l1_demo_lake` with the REST service's address `http://course-lake-rest:8181`, the warehouse `s3://course-warehouse/`, and the object store's address `http://course-lake-silo:9000` and keys; `CREATE DATABASE` and `CREATE TABLE` in that Catalog; and, on a first run, the `INSERT` of the ten orders, or on a later run a comment that the table already holds them. These addresses are names on the course's Docker network, where Doris runs: from the notebook the same services answer on 127.0.0.1:51900 and 127.0.0.1:51818.

The cell keeps the table's full name, `dw_course_l1_demo_lake.dw_course_l1_demo.orders`, in `source` (with your database name if you set `DW_DATABASE`).

## 1. Find the table through the Catalog

A **Catalog** is Doris's entry to another system's metadata: Doris's own tables are in the Catalog `internal`, and the lake table is in the Catalog that the preparation created. A table outside `internal` is named `catalog.database.table`. `SHOW CATALOGS`, `SHOW DATABASES FROM`, and `SHOW TABLES FROM` walk down that path, and `SHOW CREATE TABLE` shows what kind of table it is and where its files are.

In [ ]:
catalog, database, _ = source.split(".")
lab.sql("SHOW CATALOGS", title="The Catalogs of this Doris", columns=["CatalogName", "Type", "IsCurrent", "Comment"])
lab.sql(f"SHOW DATABASES FROM {catalog}", title="The databases in the lake Catalog")
lab.sql(f"SHOW TABLES FROM {catalog}.{database}", title="The tables in the lake database")
definition = lab.sql(f"SHOW CREATE TABLE {source}", title="What kind of table it is")
expect("ENGINE=ICEBERG_EXTERNAL_TABLE" in definition.iloc[0, 1], True, title="The table is an Iceberg table in the lake")

**Expected result**

`SHOW CATALOGS` lists `internal`, type `internal`, which is current, and `dw_course_l1_demo_lake`, type `iceberg`; Catalogs of other lab databases, if you made any, appear too. The lake Catalog holds the databases `dw_course_l1_demo`, `information_schema`, and `mysql`, and the database holds one table, `orders`. Its definition says `ENGINE=ICEBERG_EXTERNAL_TABLE` and `LOCATION 's3://course-warehouse/dw_course_l1_demo/orders'`, with Iceberg properties such as `"write.parquet.compression-codec" = "zstd"`, and the check is green. The columns are nullable, as Iceberg columns usually are, unlike the `NOT NULL` columns of the course's internal tables.

## 2. Read the table's snapshots and files

An Iceberg table is metadata over data files. Each commit, such as an `INSERT`, writes new files and a new **snapshot**: the list of files that make up the table at that moment. A query reads the files of the current snapshot. Doris exposes Iceberg's metadata as tables named after the table with a suffix: `$snapshots` lists the snapshots, and `$files` the data files of the current snapshot.

In [ ]:
lab.sql(f"SELECT committed_at, snapshot_id, parent_id, operation FROM {source}$snapshots",
        title="The snapshots of the lake table")
files = lab.sql(f"SELECT file_path, file_format, record_count, file_size_in_bytes FROM {source}$files",
                title="The data files of the current snapshot")
data_file = files["file_path"][0]

**Expected result**

One snapshot with `operation` `append` and no parent: the preparation's `INSERT` on its first run. One data file, `s3://course-warehouse/dw_course_l1_demo/orders/data/<random name>.zstd.parquet`, format `PARQUET`, 10 records, about 4.3 KB. The cell keeps its path in `data_file` for step 5. Another `INSERT` would add a second snapshot and a second file; Doris would then read both.

## 3. Query the lake table directly

Through the Catalog, Doris reads the table's metadata from the REST service, plans which files to read, reads them from Silo, and computes in its own BE. The result is in Doris's memory only: nothing is stored in an internal table. Expect ten orders and 12220.60.

In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10, "12220.60")],
       title="The lake table holds ten orders, 12220.60")
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1", title="The plan of a lake table query");
lab.sql(f"SELECT order_id, customer_id, order_date, order_amount FROM {source} ORDER BY order_id",
        title="Ten orders in the lake");

**Expected result**

A green check, then the plan: its scan is `VICEBERG_SCAN_NODE` with the table's full name, the predicate `order_id = 1`, `inputSplitNum=1, totalFileSize=4349`, and `icebergPredicatePushdown` with the same condition: Doris hands the filter to Iceberg, which can skip files whose column statistics rule it out. Then the ten orders 1–5 and 80–84.

## 4. Join an internal table and load the lake table

Create `customers_sample` in Doris, a Unique Key table with one row per customer, and join the lake orders to it on `customer_id`: each order matches exactly one customer, so the join keeps ten rows and 12220.60. Then load the six order columns into the internal table `orders_from_lake` with `INSERT INTO ... SELECT` and compare it with the lake table row by row. The internal table is a copy of today's lake data: it does not follow later changes in the lake.

In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")], title="The join keeps ten orders, 12220.60")
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("CREATE TABLE SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows(), title="orders_from_lake matches the lake table row by row")
lab.sql(f"""SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, ENGINE FROM {catalog}.information_schema.tables WHERE TABLE_NAME = 'orders'
UNION ALL
SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, ENGINE FROM internal.information_schema.tables
WHERE TABLE_SCHEMA = '{lab.database}' AND TABLE_NAME = 'orders_from_lake'""", title="The lake table and its internal copy");

**Expected result**

The table definition of `orders_from_lake`, two green checks, and two rows: the lake table `orders` with `ENGINE` `iceberg` in the lake Catalog, and `orders_from_lake` with `ENGINE` `Doris` in `internal`. The data is the same; who stores it differs.

## 5. Read the data file without its table: the S3 table function

A **table function** returns rows like a table. `S3(...)` reads files in S3-compatible object storage: you give it the path of a file, or of several with a `*`, the format, and the address and keys of the object store, and it reads exactly those files. Point it at the lake table's data file from step 2. `DESC FUNCTION` shows the columns that Doris reads from the Parquet file itself. Then load the file into `orders_from_file` with `INSERT INTO ... SELECT` and compare.

The function knows nothing of Iceberg: it does not read snapshots. If the lake table later changed, for example with rows deleted in a new snapshot, this old file would still hold them; the Catalog reads the files of the current snapshot.

In [ ]:
from dw_course.lakehouse import ACCESS_KEY, SECRET_KEY
file_function = f'''S3(
    "uri" = "{data_file}",
    "format" = "parquet",
    "s3.endpoint" = "http://course-lake-silo:9000", "s3.region" = "us-east-1",
    "s3.access_key" = "{ACCESS_KEY}", "s3.secret_key" = "{SECRET_KEY}",
    "use_path_style" = "true"
)'''
lab.sql(f"DESC FUNCTION {file_function}", title="The columns of the Parquet file", columns=["Field", "Type", "Null"])
lab.sql(f"SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM {file_function}",
        title="The file read directly")
lab.execute("DROP TABLE IF EXISTS orders_from_file")
lab.execute(order_ddl("orders_from_file"))
load_file = f"INSERT INTO orders_from_file ({','.join(ORDER_COLUMNS)})\nSELECT {','.join(ORDER_COLUMNS)}\nFROM {file_function}"
show_sql("Load the file into an internal table", load_file)
lab.execute(load_file)
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_file ORDER BY order_id"),
       history_rows(), title="orders_from_file matches the lake table row by row")

**Expected result**

Six columns from the file, the types that Doris read from Parquet: `bigint`, `bigint`, `date`, `decimal(18,2)`, `int`, and `text`, all nullable. The file read directly gives 10 orders and 12220.60, the `INSERT INTO ... SELECT` is shown, and the check is green: the file is the table's data. `use_path_style` puts the bucket name into the URL's path, as Silo expects.

## 6. Practice: the second day, with missing customers

For the lake orders of 2013-01-02, count the orders, total their amounts, and count the orders whose customer is missing from `customers_sample`, with a `LEFT JOIN`. Expect 5 orders, 8276.40, and 0 missing customers. Then compare the plans of the same date filter on the lake table and on `orders_from_lake`: which scan node reads each, and what does each say it will read?

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
query = f"""SELECT COUNT(*) AS orders, SUM(o.order_amount) AS amount,
SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers
FROM {source} o LEFT JOIN customers_sample c ON o.customer_id=c.customer_id
WHERE o.order_date='2013-01-02'"""
lab.sql(query, title="Customer matches for the second day's lake orders");
expect(lab.query(query), [(5,"8276.40",0)], title="Five orders, 8276.40, no customer missing")
lab.sql("EXPLAIN SELECT * FROM orders_from_lake WHERE order_date='2013-01-02'", title="Internal table scan");
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_date='2013-01-02'", title="Lake table scan");

**Expected result**

One row: 5 orders, 8276.40, 0 missing customers; a `LEFT JOIN` keeps an order without a customer and leaves its customer columns `NULL`, so this count would find one. The internal table's plan scans with `VOlapScanNode`, `partitions=1/1` and `tablets=1/1`: the tablet that Doris stores. The lake table's plan scans with `VICEBERG_SCAN_NODE`, `inputSplitNum=1` and `totalFileSize=4349`: a file in object storage that it found through the table's metadata, with the date filter pushed down to Iceberg.

## You are done when

- you can name the lake table's Catalog, database, and table, and say what `ENGINE=ICEBERG_EXTERNAL_TABLE` and its `LOCATION` tell you;
- you have read its snapshot and its data file, and can explain why a query of the table reads only the current snapshot's files;
- the lake table, the join, `orders_from_lake`, and `orders_from_file` all give ten orders and 12220.60, and you can say which of them is stored by Doris.

## Next

Open [Quiz 4: Querying External Data and Joining Lake Tables](quiz04_internal_files_and_lake_tables.ipynb), then continue to [Module 5: Batch and Continuous Data Ingestion](../module05-ingestion/course.md).

To pause the lake and keep its data, run this from the course directory:

```bash
docker compose --project-name doris-warehousing-lake --file environments/lakehouse/compose.yml stop
```

Rerun this Lab from its setup through **Prepare the lake** to resume it. The Module 4–5 extension uses the same lake data; keep the volumes until you finish it.
